# 05 — Cotejo de tatuajes: Necropsias vs Fichas de Búsqueda

Comparación de la columna `DESCRIPCIÓN TATUAJES` de la tabla `proyecto_mna.fichas_busqueda.registro_necropsias_2026` contra el índice de Vector Search `texto_completo_sin_features`, que contiene descripciones de tatuajes de fichas de personas desaparecidas (`texto_tatuajes`). Para cada necropsia con tatuajes reales, se consulta el índice con similitud semántica (k-NN) y se recuperan las fichas más parecidas, mostrando lado a lado las descripciones de tatuajes para inspección forense.

In [0]:
# ============================================================================
# Cell 1 — Setup y carga de necropsias con descripciones de tatuajes reales
# ============================================================================
import json
import pandas as pd
from databricks.sdk import WorkspaceClient

# --- Cliente del workspace (autenticación automática) ---
w = WorkspaceClient()

# --- Configuración del índice de Vector Search ---
# Índice: proyecto_mna.fichas_busqueda.texto_completo_sin_features
# Endpoint: forensic (ONLINE)
# Primary key: id_ficha
# Embedding: texto_completo → modelo databricks-qwen3-embedding-0-6b
# Columna de tatuajes: texto_tatuajes (disponible como columna de retorno)
INDEX_NAME = "proyecto_mna.fichas_busqueda.texto_completo_sin_features"
COLUMNS = ["id_ficha", "sexo", "edad_desaparicion", "texto_tatuajes", "texto_completo"]
TOP_K = 10  # Número de candidatos por necropsia

# --- Cargar necropsias con descripciones de tatuajes reales ---
# Se excluyen registros sin tatuajes: "NO APLICA", "N/A", vacíos, NULL
necropsias_df = spark.sql("""
    SELECT 
        `#` AS id_necropsia,
        SEXO AS sexo_necropsia,
        `EDAD (AÑOS, MESES, SEMANAS, DÍAS)` AS edad_necropsia,
        `DESCRIPCIÓN TATUAJES` AS descripcion_tatuajes
    FROM proyecto_mna.fichas_busqueda.registro_necropsias_2026
    WHERE `DESCRIPCIÓN TATUAJES` IS NOT NULL 
      AND `DESCRIPCIÓN TATUAJES` != ''
      AND UPPER(TRIM(`DESCRIPCIÓN TATUAJES`)) NOT IN ('NO APLICA', 'N/A', 'SIN_DATO')
""").toPandas()

print(f"✅ Necropsias con tatuajes reales: {len(necropsias_df)}")
for _, row in necropsias_df.iterrows():
    print(f"   [#{row['id_necropsia']:.0f}] {row['sexo_necropsia']} | {row['edad_necropsia']} | {row['descripcion_tatuajes'][:100]}...")

In [0]:
# ============================================================================
# Cell 2 — Consulta al índice Vector Search por cada necropsia
# ============================================================================
# Para cada necropsia con tatuajes reales, se consulta el índice
# texto_completo_sin_features usando la descripción de tatuajes como query_text.
# Se aplica un filtro de blocking por sexo para reducir el espacio de búsqueda.
# Se recuperan las columnas id_ficha, sexo, edad_desaparicion, texto_tatuajes
# y texto_completo para comparación.


def buscar_candidatos_tatuajes(query_text, k=10, filters=None):
    """
    Busca fichas candidatas por similitud semántica en el índice de Vector Search.

    Parámetros
    ----------
    query_text : str
        Descripción de tatuajes de la necropsia (texto de consulta).
    k : int
        Número de candidatos a retornar (top-k).
    filters : dict, opcional
        Filtros de blocking en formato diccionario para endpoint Standard.
        Ejemplo: {"sexo": "HOMBRE"}

    Retorna
    -------
    pandas.DataFrame
        DataFrame con columnas: score, id_ficha, sexo, edad_desaparicion,
        texto_tatuajes, texto_completo.
    """
    filters_json = json.dumps(filters) if filters else None
    results = w.vector_search_indexes.query_index(
        index_name=INDEX_NAME,
        columns=COLUMNS,
        query_text=query_text,
        num_results=k,
        filters_json=filters_json
    )
    col_names = [c.name for c in results.manifest.columns]
    rows = results.result.data_array
    if not rows:
        return pd.DataFrame(columns=col_names)
    return pd.DataFrame(rows, columns=col_names)


# --- Mapeo de sexo necropsia → valor del filtro del índice ---
# El índice tiene la columna 'sexo' con valores: HOMBRE, MUJER
def mapear_sexo_filter(sexo_necropsia):
    """Mapea el sexo de la necropsia al formato del filtro del índice."""
    s = str(sexo_necropsia).strip().upper()
    if "HOMBRE" in s:
        return "HOMBRE"
    elif "MUJER" in s:
        return "MUJER"
    return None  # INDETERMINADO → sin filtro de sexo


# --- Ejecutar comparación para cada necropsia ---
todos_resultados = []

for _, nec in necropsias_df.iterrows():
    id_nec = int(nec["id_necropsia"])
    sexo_filter = mapear_sexo_filter(nec["sexo_necropsia"])
    query_text = nec["descripcion_tatuajes"]

    # Construir filtro de blocking
    filters = {"sexo": sexo_filter} if sexo_filter else None

    print(f"\n{'='*70}")
    print(f"Necropsia #{id_nec} | {nec['sexo_necropsia']} | {nec['edad_necropsia']}")
    print(f"Filtro: {filters}")
    print(f"Query: {query_text[:120]}...")
    print(f"{'='*70}")

    candidatos = buscar_candidatos_tatuajes(query_text, k=TOP_K, filters=filters)

    for rank, (_, row) in enumerate(candidatos.iterrows(), 1):
        score = row["score"]
        resultado = {
            "id_necropsia": id_nec,
            "sexo_necropsia": nec["sexo_necropsia"],
            "edad_necropsia": nec["edad_necropsia"],
            "tatuajes_necropsia": query_text,
            "rank": rank,
            "score": round(float(score), 4),
            "id_ficha": row["id_ficha"],
            "sexo_ficha": row["sexo"],
            "edad_ficha": row["edad_desaparicion"],
            "tatuajes_ficha": row["texto_tatuajes"] if row["texto_tatuajes"] else "",
        }
        todos_resultados.append(resultado)

        print(f"  [{rank}] score={score:.4f} | {row['id_ficha']} | {row['sexo']} | edad={row['edad_desaparicion']}")
        tatuajes_str = str(row["texto_tatuajes"]) if row["texto_tatuajes"] else "(sin tatuajes)"
        print(f"      tatuajes: {tatuajes_str[:150]}...")

# --- Consolidar resultados en un DataFrame ---
resultados_df = pd.DataFrame(todos_resultados)

# --- Enriquecer con la ruta del archivo (PDF/imagen) desde fichas_persona_silver ---
rutas_df = spark.sql("""
    SELECT id_ficha, ruta
    FROM proyecto_mna.fichas_busqueda.fichas_persona_silver
""").toPandas()

resultados_df = resultados_df.merge(rutas_df, on="id_ficha", how="left")
# Extraer solo el nombre del archivo de la ruta completa
resultados_df["nombre_archivo"] = resultados_df["ruta"].apply(
    lambda p: p.split("/")[-1] if pd.notna(p) else "(sin archivo)"
)

print(f"\n✅ Total de comparaciones: {len(resultados_df)} ({len(necropsias_df)} necropsias × {TOP_K} candidatos)")

In [0]:
# ============================================================================
# Cell 3 — Tabla comparativa: tatuajes necropsia vs ficha
# ============================================================================
# Muestra la tabla completa de comparaciones con las descripciones de
# tatuajes lado a lado para inspección forense.

# Seleccionar y renombrar columnas para display
display_cols = resultados_df[[
    "id_necropsia", "rank", "score",
    "sexo_necropsia", "sexo_ficha",
    "edad_necropsia", "edad_ficha",
    "nombre_archivo",
    "tatuajes_necropsia", "tatuajes_ficha"
]].copy()

display_cols.columns = [
    "Necropsia #", "Rank", "Score",
    "Sexo (nec)", "Sexo (ficha)",
    "Edad (nec)", "Edad (ficha)",
    "Archivo PDF",
    "Tatuajes (necropsia)", "Tatuajes (ficha)"
]

print("Tabla comparativa de tatuajes: Necropsias vs Fichas de búsqueda")
print(f"{len(resultados_df)} comparaciones | {resultados_df['id_necropsia'].nunique()} necropsias | top-{TOP_K} por necropsia")
print()
display(display_cols)